In [45]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
!pip install pdfplumber langchain-core

In [47]:
import sys
sys.path.append("/content/drive/MyDrive/rag-mmlu-ewha")

In [48]:
# ==========================
# 1) PDF → TEXT 추출
# ==========================

import os
import pdfplumber

def extract_text_from_pdf(pdf_path: str) -> str:
    """PDF 전체 텍스트 추출"""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"{pdf_path} not found: 현재 경로 = {os.getcwd()}")

    lines = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            txt = page.extract_text() or ""
            txt = txt.replace("\u00A0", " ")
            lines.append(txt)
        print(f"총 {len(pdf.pages)} 페이지에서 텍스트 추출 완료")

    return "\n".join(lines)

In [49]:
pdf_path = "/content/drive/MyDrive/rag-mmlu-ewha/data/ewha.pdf"

# 실제 PDF에서 텍스트 뽑아서 변수에 담기
raw_text = extract_text_from_pdf(pdf_path)

print(raw_text[:2000]) # 앞부분 일부 보기

총 54 페이지에서 텍스트 추출 완료
이화여자대학교 학칙
1946. 8. 15. 제정
2017. 8. 16. 개정
제1장 총칙
제1조(목적) 본교는 대한민국의 교육이념과 기독교정신을 바탕으로 하여 학술의 깊은 이론과
그 광범하고 정밀한 응용방법을 교수․연구하며, 인격을 도야하여 국가와 인류사회의 발전에
공헌할 수 있는 지도여성을 양성함을 목적으로 한다.
제2조(명칭) 본교는 이화여자대학교라 부른다.
제3조(위치) 본교는 서울특별시 서대문구 이화여대길 52에 둔다. (개정 2013.2.25.)
제2장 편제
제4조(대학 및 대학원) ① 본교에는 다음 각 호의 대학을 둔다.
1. 인문과학대학, 사회과학대학, 자연과학대학, 엘텍공과대학, 음악대학, 조형예술대학, 사범
대학, 경영대학, 신산업융합대학, 의과대학, 간호대학, 약학대학, 스크랜튼대학(이하 “각
대학”이라 한다) (개정 2016.6.16.)
2. 호크마(HOKMA)교양대학
② 본교에는 대학원, 국제대학원, 통역번역대학원, 경영전문대학원, 법학전문대학원, 교육대
학원, 디자인대학원, 사회복지대학원, 신학대학원, 정책과학대학원, 공연예술대학원, 임상보
건융합대학원, 임상치의학대학원, 외국어교육특수대학원을 둔다(이하 “각 대학원”이라 한다).
(개정 2016.6.16., 2017.5.15.)
[전문개정 2015.11.27.]
제5조(학부․학과․전공 및 정원) ① 각 대학, 학부, 학과, 전공 및 모집단위별 입학정원은 별표
1과 같다. (개정 2015.5.8., 2016.2.16., 2016.2.26., 2016.5.19., 2017.5.4., 2017.5.1
5.)
② 모집단위별 입학정원의 일부는 입학전형에 따라 2개 이상의 모집단위를 통합하여 모집
할 수 있다. (개정 1999.2.9., 2017.5.15.)
③ 제2항에 따라 통합된 모집단위로 입학한 학생과 대학 또는 학부 등 광역화된 모집단위
로 입학한 학생에 대하여는 일정한 학기와 학점을 이수한 후에 총장의 승인을 얻어 이수할
전공을 결정하게 하되 이에 필

In [50]:
out_txt_path = "/content/drive/MyDrive/rag-mmlu-ewha/data/ewha.txt"

with open(out_txt_path, "w", encoding="utf-8") as f:
    f.write(raw_text)

print("저장 완료:", out_txt_path)

저장 완료: /content/drive/MyDrive/rag-mmlu-ewha/data/ewha.txt


In [71]:
# ==========================
# 2) 구조 파싱 (제X장, [별표 X])
# ==========================

import re

def parse_ewha_structure(text: str):
    """
    장(제X장)은 정규식으로 파싱.
    별표(Appendix)는 PDF 맨 뒤 chunk에서
    '네가 지정한 Appendix 헤더 리스트'를 기준으로 하드코딩 파싱.
    """

    # ----------------------------
    # 1) Chapter 파싱
    # ----------------------------
    chapter_pattern = r"제\d+장"
    chapter_markers = re.findall(chapter_pattern, text)
    chapter_splits = re.split(chapter_pattern, text)

    # ----------------------------
    # 2) Appendix 파싱 (하드코딩)
    # ----------------------------
    last_chunk = chapter_splits[-1]  # 부칙 이후 텍스트 전체

    APPENDIX_HEADERS = [
        "[별표 1] (개정 2017.5.4., 2017.5.15.)",
        "[별표 1] (개정 2016.2.26., 2016.5.19., 2017.5.4., 2017.5.15.)",
        "[별표 1] (개정 2015.5.8., 2016.2.16., 2016.5.19.)",
        "[별표 1] (개정 2015.5.8.)",
        "[별표 1] (개정 2014.9.26.)",
        "[별표 1] (개정 2013.11.20.)",
        "[별표 2] (개정 2014.11.21., 2015.9.18., 2016.2.16., 2016.3.31., 2017.2.8.)",
        "[별표 3] (개정 2014.5.15., 2016.6.16.)"
    ]

    appendix_markers = []
    appendix_splits = []

    remaining_text = last_chunk

    for header in APPENDIX_HEADERS:
        if header in remaining_text:
            idx = remaining_text.index(header)
            appendix_markers.append(header)
            appendix_splits.append(remaining_text[:idx])
            remaining_text = remaining_text[idx + len(header):].strip()

    # 마지막 appendix의 텍스트 추가
    appendix_splits.append(remaining_text)

    return chapter_markers, chapter_splits, appendix_markers, appendix_splits


In [72]:
chapter_markers, chapter_splits, appendix_markers, appendix_splits = parse_ewha_structure(raw_text)


In [73]:
print("Chapters:", chapter_markers) # Chapter 구조 탐지 확인
print("총 장 수:", len(chapter_markers))

print("split[0] (서문):") # 첫 번째 분할(제1장 이전), 두 번째 분할(제1장 내용) 확인
print(chapter_splits[0][:500])

print("\n==== 제1장 내용 미리보기 ====")
print(chapter_markers[0])
print(chapter_splits[1][:500])

print("Appendix markers:", appendix_markers)

Chapters: ['제1장', '제2장', '제3장', '제4장', '제5장', '제6장', '제7장', '제8장', '제9장', '제10장', '제11장', '제12장', '제13장', '제14장', '제15장', '제16장', '제17장']
총 장 수: 17
split[0] (서문):
이화여자대학교 학칙
1946. 8. 15. 제정
2017. 8. 16. 개정


==== 제1장 내용 미리보기 ====
제1장
 총칙
제1조(목적) 본교는 대한민국의 교육이념과 기독교정신을 바탕으로 하여 학술의 깊은 이론과
그 광범하고 정밀한 응용방법을 교수․연구하며, 인격을 도야하여 국가와 인류사회의 발전에
공헌할 수 있는 지도여성을 양성함을 목적으로 한다.
제2조(명칭) 본교는 이화여자대학교라 부른다.
제3조(위치) 본교는 서울특별시 서대문구 이화여대길 52에 둔다. (개정 2013.2.25.)

Appendix markers: ['[별표 1] (개정 2017.5.4., 2017.5.15.)', '[별표 1] (개정 2016.2.26., 2016.5.19., 2017.5.4., 2017.5.15.)', '[별표 1] (개정 2015.5.8., 2016.2.16., 2016.5.19.)', '[별표 1] (개정 2015.5.8.)', '[별표 1] (개정 2014.9.26.)', '[별표 1] (개정 2013.11.20.)', '[별표 2] (개정 2014.11.21., 2015.9.18., 2016.2.16., 2016.3.31., 2017.2.8.)', '[별표 3] (개정 2014.5.15., 2016.6.16.)']


In [74]:
from langchain_core.documents import Document
import re

#############################################
# STEP 0 — 서식 제거 함수
#############################################
def remove_forms_section(text: str) -> str:
    form_pattern = r"(서식 제1호)"
    m = re.search(form_pattern, text)
    if m:
        return text[:m.start()]
    return text


#############################################
# STEP 1 — 내부 개행/헤더/페이지번호 제거
#############################################
def clean_text_block(text: str) -> str:
    text = remove_forms_section(text)
    text = re.sub(r"\d+\s*-\s*\d+\s*-\s*\d+", "", text)
    text = text.replace("이화여자대학교 학칙", "")
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


#############################################
# STEP 2 — Document 생성 (수정된 버전)
#############################################
def make_document(doc_id: int, section: str, text: str) -> Document:

    # "제5장" → 5 (article_no)
    m = re.match(r"제(\d+)장", section)
    article_no = int(m.group(1)) if m else None

    return Document(
        id=doc_id,
        page_content=text,
        metadata={
            "doc_id": doc_id,      # 숫자 ID
            "section": section,     # 사람이 읽는 제목
            "article_no": article_no,
            "source": "ewha_regulations"
        }
    )


#############################################
# STEP 3 — Document 변환 (수정된 버전)
#############################################
def convert_to_documents(chapter_markers, chapter_splits, appendix_markers, appendix_splits):
    docs = []

    # -----------------------------
    # 1) preamble (제1장 이전)
    # -----------------------------
    preamble_text = clean_text_block(chapter_splits[0])
    docs.append(
        make_document(
            1,               # doc_id=1
            "서문",           # section 이름 변경
            preamble_text
        )
    )

    # -----------------------------
    # 2) 각 "제X장" 문서들
    # -----------------------------
    doc_counter = 2   # 2부터 시작 (1은 preamble)

    for marker, split_text in zip(chapter_markers, chapter_splits[1:]):
        cleaned = clean_text_block(split_text)
        full_text = f"{marker}\n{cleaned}"
        docs.append(make_document(doc_counter, marker, full_text))
        doc_counter += 1

    # -----------------------------
    # 3) 별표 문서들
    # -----------------------------
    for marker, split_text in zip(appendix_markers, appendix_splits[1:]):
        cleaned = clean_text_block(split_text)
        full_text = f"{marker}\n{cleaned}"
        docs.append(make_document(doc_counter, marker, full_text))
        doc_counter += 1

    return docs


In [75]:
documents = convert_to_documents(
    chapter_markers, chapter_splits,
    appendix_markers, appendix_splits
)

In [76]:
documents

[Document(id='1', metadata={'doc_id': 1, 'section': '서문', 'article_no': None, 'source': 'ewha_regulations'}, page_content='1946. 8. 15. 제정\n2017. 8. 16. 개정'),
 Document(id='2', metadata={'doc_id': 2, 'section': '제1장', 'article_no': 1, 'source': 'ewha_regulations'}, page_content='제1장\n총칙\n제1조(목적) 본교는 대한민국의 교육이념과 기독교정신을 바탕으로 하여 학술의 깊은 이론과\n그 광범하고 정밀한 응용방법을 교수․연구하며, 인격을 도야하여 국가와 인류사회의 발전에\n공헌할 수 있는 지도여성을 양성함을 목적으로 한다.\n제2조(명칭) 본교는 이화여자대학교라 부른다.\n제3조(위치) 본교는 서울특별시 서대문구 이화여대길 52에 둔다. (개정 2013.2.25.)'),
 Document(id='3', metadata={'doc_id': 3, 'section': '제2장', 'article_no': 2, 'source': 'ewha_regulations'}, page_content='제2장\n편제\n제4조(대학 및 대학원) ① 본교에는 다음 각 호의 대학을 둔다.\n1. 인문과학대학, 사회과학대학, 자연과학대학, 엘텍공과대학, 음악대학, 조형예술대학, 사범\n대학, 경영대학, 신산업융합대학, 의과대학, 간호대학, 약학대학, 스크랜튼대학(이하 “각\n대학”이라 한다) (개정 2016.6.16.)\n2. 호크마(HOKMA)교양대학\n② 본교에는 대학원, 국제대학원, 통역번역대학원, 경영전문대학원, 법학전문대학원, 교육대\n학원, 디자인대학원, 사회복지대학원, 신학대학원, 정책과학대학원, 공연예술대학원, 임상보\n건융합대학원, 임상치의학대학원, 외국어교육특수대학원을 둔다(이하 “각 대학원”이라 한다).\n(개정 2016.6.16., 2017.5.15

In [62]:
'''
ewha.pdf에서 실제로 나타나는 문제들
1. 문장이 페이지 넘어가면서 끊김
2. 개행
3. 개정 날짜가 두줄로 깨지는 경우 (개정\n2016)
4. 페이지 번호 2-2-17
5. 조항 사이의 개행이 불필요하게 3-4줄로 늘어나는 경우
6. 제목? 장 밑에 개행 정리
7. 들여쓰기 간격
'''

'\newha.pdf에서 실제로 나타나는 문제들 \n1. 문장이 페이지 넘어가면서 끊김 \n2. 개행\n3. 개정 날짜가 두줄로 깨지는 경우 (개정\n2016)\n4. 페이지 번호 2-2-17\n5. 조항 사이의 개행이 불필요하게 3-4줄로 늘어나는 경우\n6. 제목? 장 밑에 개행 정리\n7. 들여쓰기 간격\n'

In [77]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def create_llm_cleanup_chain(llm):
    prompt = ChatPromptTemplate.from_template("""
다음 텍스트는 '이화여자대학교 학칙' PDF에서 추출된 원문입니다.
PDF 추출 과정에서 줄바꿈, 들여쓰기, 개행(\n), 공백, 조항 분리, 개정일자 형식 등이 깨지는 문제가 있습니다.

아래 규칙에 따라 텍스트를 올바른 문장이 되도록 정제하세요.

=========================
**정제 규칙**
=========================

1) **문장 중간에 끊긴 줄바꿈 제거**
   - 예: "인류사회의 발전에\n공헌할 수 있는" → "인류사회의 발전에 공헌할 수 있는"

2) **"(개정 yyyy.mm.dd.)"이나 "(신설 yyyy.mm.dd.)" 형식은 한 줄로 유지**
   - 예: "(개정\n2016.6.16.)" → "(개정 2016.6.16.)"
   - 예: "(신설\n2016.6.16.)" → "(신설 2016.6.16.)"

3) **페이지 번호, 머리글, 꼬리글 제거**
   - 예: "2 - 2 - 17" 같은 숫자 패턴 삭제
   - "이화여자대학교 학칙" 같은 반복 헤더 삭제

4) **불필요한 공백 제거**
   - 문장 앞뒤 공백 제거
   - 중복 공백은 하나로 축소
   - "교수․연구하며" 같은 특수문자는 유지
5) **의미를 절대 변경하지 않음**
   - 단어 추가 금지
   - 조항 번호 변경 금지

=========================
출력 형식
=========================
- 오직 정제된 텍스트만 출력
- JSON, 설명, 메모 X
- 불필요한 줄바꿈 없이 조항 단위로만 구분

=========================
정제할 텍스트:
{doc}

=========================
정제된 최종 텍스트:
""")

    chain = (
        {"doc": lambda x: x.page_content}
        | prompt
        | llm.with_config(temperature=0)
        | StrOutputParser()
    )
    return chain


In [64]:
!pip install -q langchain langchain-community openai

In [78]:
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(
    api_key="up_l9ItU36BE9QJ8wiALOb7DnkedWilS",
    base_url="https://api.upstage.ai/v1",
    model="solar-mini",   # 또는 solar-pro
    temperature=0
)

llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x797b49546990>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x797b418ffec0>, model_name='solar-mini', temperature=0.0, model_kwargs={}, openai_api_key='up_l9ItU36BE9QJ8wiALOb7DnkedWilS', openai_api_base='https://api.upstage.ai/v1', openai_proxy='')

In [79]:
# 정제 체인 생성
llm_chain = create_llm_cleanup_chain(llm)

# 문서 정제 실행
cleaned_documents = []
for i, doc in enumerate(documents):
    try:
        print(f"Cleaning document {i+1}/{len(documents)}: {doc.metadata['section']}")
        cleaned_text = llm_chain.invoke(doc)

        # 새 Document 생성 (metadata 동일)
        new_doc = Document(
            id=doc.id,
            page_content=cleaned_text,
            metadata=doc.metadata
        )
        cleaned_documents.append(new_doc)

    except Exception as e:
        print(f"Error on doc {i}: {e}")
        # 실패 시 원본 사용
        cleaned_documents.append(doc)

print("LLM 정제 완료!")


Cleaning document 1/26: 서문
Cleaning document 2/26: 제1장
Cleaning document 3/26: 제2장
Cleaning document 4/26: 제3장
Cleaning document 5/26: 제4장
Cleaning document 6/26: 제5장
Cleaning document 7/26: 제6장
Cleaning document 8/26: 제7장
Cleaning document 9/26: 제8장
Cleaning document 10/26: 제9장
Cleaning document 11/26: 제10장
Cleaning document 12/26: 제11장
Cleaning document 13/26: 제12장
Cleaning document 14/26: 제13장
Cleaning document 15/26: 제14장
Cleaning document 16/26: 제15장
Cleaning document 17/26: 제16장
Cleaning document 18/26: 제17장
Cleaning document 19/26: [별표 1] (개정 2017.5.4., 2017.5.15.)
Cleaning document 20/26: [별표 1] (개정 2016.2.26., 2016.5.19., 2017.5.4., 2017.5.15.)
Cleaning document 21/26: [별표 1] (개정 2015.5.8., 2016.2.16., 2016.5.19.)
Cleaning document 22/26: [별표 1] (개정 2015.5.8.)
Cleaning document 23/26: [별표 1] (개정 2014.9.26.)
Cleaning document 24/26: [별표 1] (개정 2013.11.20.)
Cleaning document 25/26: [별표 2] (개정 2014.11.21., 2015.9.18., 2016.2.16., 2016.3.31., 2017.2.8.)
Cleaning document 26/26: [별

In [80]:
# === 정제된 문서 샘플 확인 ===
i = 18   # 보고 싶은 문서 번호
print("===== BEFORE =====")
print(documents[i].page_content[:800])

print("\n===== AFTER =====")
print(cleaned_documents[i].page_content[:800])


===== BEFORE =====
[별표 1] (개정 2017.5.4., 2017.5.15.)
학부, 학과, 전공별 입학정원(2019학년도) (제5조 관련)
대학 학부/학과/전공 입학정원
국어국문학과
중어중문학과
불어불문학과
독어독문학과 299
인문과학대학 사학과
철학과
기독교학과
영어영문학부 91
소계 390
정치외교학과
행정학과
경제학과
문헌정보학과
290
사회학과
사회과학대학
사회복지학과
심리학과
소비자학과
커뮤니케이션·미디어학부 79
소계 369
수학과
통계학과 118
물리학과
자연과학대학 화학생명분자과학부 149
화학·나노과학전공
생명과학전공
소계 267
휴먼기계바이오공학부 110
소프트웨어학부 105
컴퓨터공학전공
사이버보안전공
차세대기술공학부 155
전자전기공학전공
화학신소재공학전공
엘텍공과대학
식품공학전공
미래사회공학부 144
기후·에너지시스템공학전공
환경공학전공
건축도시시스템공학전공
건축학전공
소계 514
건반악기과
관현악과
성악과 164
음악대학 작곡과
한국음악과
무용과 38
소계 202
조형예술학부 113
동양화전공
서양화전공
조소전공
도자예술전공
조형예술대학
디자인학부 80
섬유·패션학부 47
섬유예술전공
패션디자인전공
소계 240
(계속)
대학 학부/학과/전공 입학정원
교육학과 27
유아교육과 29
초등교육과 39
교육공학과 30
특수교육과 35
유아특수교육전공
초등특수교육전공
중등특수교육전공
영어교육과 37
사회과교육과 71
사범대학 역사교육전공
사회교육전공
지리교육전공
국어교육과 27
과학교육과 86
물리교육전공
화학교육전공
생물교육전공
지구과학교육전공
수학교육과 27
소계 408
경영학부 116
경영대학
소계 11

===== AFTER =====
[별표 1] (개정 2017.5.4., 2017.5.15.) 학부, 학과, 전공별 입학정원(2019학년도) (제5조 관련) 대학 학부/학과/전공 입학정원 국어국문학과 중어중문학과 불어불문학과 독어독문학과 299 인문과학대학 사학과 철학과 기독교학과 영어영문학부 91 소계 390 정치외교학

In [81]:
import json
from pathlib import Path

def save_jsonl(doc_list, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as f:
        for doc in doc_list:
            row = {
                "doc_id": doc.metadata["doc_id"],
                "section": doc.metadata["section"],
                "article_no": doc.metadata["article_no"],
                "text": doc.page_content
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"JSONL 저장 완료: {output_path}")

save_jsonl(
    cleaned_documents,
    "/content/drive/MyDrive/rag-mmlu-ewha/data/ewha_clean.jsonl"
)


JSONL 저장 완료: /content/drive/MyDrive/rag-mmlu-ewha/data/ewha_clean.jsonl
